# Export Prometheus TSDB → CSV

Pulls every metric used in `experiment_statistical.ipynb` from the backup Prometheus  
(`http://localhost:9091`) and saves them as CSV files in `notebooks/prometheus_export/`.

Run this once while `prom-restore` Docker container is alive.  
After that the data is safe on disk regardless of Minikube/Prometheus state.

In [4]:
import json
import requests
import pandas as pd
from pathlib import Path

PROM         = 'http://localhost:9091/api/v1/query_range'
TIMESTAMPS_F = Path('../test_deployment/locusts/experiment_timestamps.json')
OUT_DIR      = Path('../notebooks/prometheus_export')
OUT_DIR.mkdir(exist_ok=True)

RUNS = [
    {'hpa': 'round_a_mw2_2', 'keda': 'round_b_mw2_2'},
    {'hpa': 'round_a_mw2_3', 'keda': 'round_b_mw2_3'},
    {'hpa': 'round_a_mw2_4', 'keda': 'round_b_mw2_4'},
]

SCENARIO_NAMES = [
    '1. Linear Ramp',
    '2. Step Function',
    '3. Sine Wave',
    '4. Multi Spike',
    '5. Random Walk',
    '6. Quiet + Burst',
]

IDLE_PAD_S = 30

CPU_UTIL_Q = (
    'sum(rate(container_cpu_usage_seconds_total'
    '{pod=~"cpu-service-.*",container!="POD"}[30s]))'
    '/sum(kube_pod_container_resource_limits'
    '{pod=~"cpu-service-.*",resource="cpu",container!="POD"})'
)

METRICS = {
    'replicas':  'kube_deployment_status_replicas_ready{deployment="cpu-service"}',
    'cpu_util':  CPU_UTIL_Q,
    'gru_pred':  'gru_predicted_cpu_util',
}

with open(TIMESTAMPS_F) as f:
    timestamps = json.load(f)

print('Output dir:', OUT_DIR.resolve())
print('Metrics to export:', list(METRICS.keys()))

Output dir: /home/compicraft/projects/diploma/notebooks/prometheus_export
Metrics to export: ['replicas', 'cpu_util', 'gru_pred']


In [5]:
def prom_range(query, start, end, step='15s'):
    r = requests.get(PROM,
                     params={'query': query, 'start': start, 'end': end, 'step': step},
                     timeout=30)
    result = r.json()['data']['result']
    if not result:
        return pd.DataFrame(columns=['ts', 'value'])
    df = pd.DataFrame(result[0]['values'], columns=['ts', 'value'])
    df['ts']    = df['ts'].astype(float)
    df['value'] = pd.to_numeric(df['value'])
    return df


rows_saved = 0

for run in RUNS:
    for role, round_key in [('hpa', run['hpa']), ('keda', run['keda'])]:
        for scenario in SCENARIO_NAMES:
            ts_info = timestamps[round_key][scenario]
            start   = ts_info['start_unix'] - IDLE_PAD_S
            end     = ts_info['end_unix']   + IDLE_PAD_S

            for metric_name, query in METRICS.items():
                df = prom_range(query, start, end, step='15s')
                df['round']    = round_key
                df['scenario'] = scenario
                df['metric']   = metric_name

                safe_scenario = scenario.replace('. ', '_').replace(' ', '_').lower()
                fname = f'{round_key}__{safe_scenario}__{metric_name}.csv'
                df.to_csv(OUT_DIR / fname, index=False)
                rows_saved += len(df)

            print(f'  saved: {round_key} / {scenario}')

print(f'\nDone. Total rows saved: {rows_saved}')
print(f'Files in {OUT_DIR}:', len(list(OUT_DIR.glob('*.csv'))))

  saved: round_a_mw2_2 / 1. Linear Ramp
  saved: round_a_mw2_2 / 2. Step Function
  saved: round_a_mw2_2 / 3. Sine Wave
  saved: round_a_mw2_2 / 4. Multi Spike
  saved: round_a_mw2_2 / 5. Random Walk
  saved: round_a_mw2_2 / 6. Quiet + Burst
  saved: round_b_mw2_2 / 1. Linear Ramp
  saved: round_b_mw2_2 / 2. Step Function
  saved: round_b_mw2_2 / 3. Sine Wave
  saved: round_b_mw2_2 / 4. Multi Spike
  saved: round_b_mw2_2 / 5. Random Walk
  saved: round_b_mw2_2 / 6. Quiet + Burst
  saved: round_a_mw2_3 / 1. Linear Ramp
  saved: round_a_mw2_3 / 2. Step Function
  saved: round_a_mw2_3 / 3. Sine Wave
  saved: round_a_mw2_3 / 4. Multi Spike
  saved: round_a_mw2_3 / 5. Random Walk
  saved: round_a_mw2_3 / 6. Quiet + Burst
  saved: round_b_mw2_3 / 1. Linear Ramp
  saved: round_b_mw2_3 / 2. Step Function
  saved: round_b_mw2_3 / 3. Sine Wave
  saved: round_b_mw2_3 / 4. Multi Spike
  saved: round_b_mw2_3 / 5. Random Walk
  saved: round_b_mw2_3 / 6. Quiet + Burst
  saved: round_a_mw2_4 / 1. Line

In [6]:
# Verify: reload one file and spot-check values
sample = pd.read_csv(OUT_DIR / 'round_a_mw2_3__1_linear_ramp__replicas.csv')
print('Sample file — round_a_mw2_3 / Linear Ramp / replicas:')
print(sample.head(10).to_string(index=False))
print(f'\nUnique replica counts: {sorted(sample["value"].unique())}')

Sample file — round_a_mw2_3 / Linear Ramp / replicas:
          ts  value         round       scenario   metric
1.776957e+09      1 round_a_mw2_3 1. Linear Ramp replicas
1.776957e+09      1 round_a_mw2_3 1. Linear Ramp replicas
1.776957e+09      1 round_a_mw2_3 1. Linear Ramp replicas
1.776957e+09      1 round_a_mw2_3 1. Linear Ramp replicas
1.776957e+09      1 round_a_mw2_3 1. Linear Ramp replicas
1.776957e+09      1 round_a_mw2_3 1. Linear Ramp replicas
1.776957e+09      1 round_a_mw2_3 1. Linear Ramp replicas
1.776957e+09      1 round_a_mw2_3 1. Linear Ramp replicas
1.776957e+09      1 round_a_mw2_3 1. Linear Ramp replicas
1.776957e+09      1 round_a_mw2_3 1. Linear Ramp replicas

Unique replica counts: [np.int64(1), np.int64(2), np.int64(4)]


In [7]:
# Print file list with row counts
files = sorted(OUT_DIR.glob('*.csv'))
print(f'{'File':<70}  {'Rows':>6}')
print('-' * 80)
for f in files:
    n = len(pd.read_csv(f))
    print(f'{f.name:<70}  {n:>6}')

File                                                                      Rows
--------------------------------------------------------------------------------
round_a_mw2_2__1_linear_ramp__cpu_util.csv                                  43
round_a_mw2_2__1_linear_ramp__gru_pred.csv                                  45
round_a_mw2_2__1_linear_ramp__replicas.csv                                  45
round_a_mw2_2__2_step_function__cpu_util.csv                                42
round_a_mw2_2__2_step_function__gru_pred.csv                                45
round_a_mw2_2__2_step_function__replicas.csv                                45
round_a_mw2_2__3_sine_wave__cpu_util.csv                                    53
round_a_mw2_2__3_sine_wave__gru_pred.csv                                    53
round_a_mw2_2__3_sine_wave__replicas.csv                                    53
round_a_mw2_2__4_multi_spike__cpu_util.csv                                  45
round_a_mw2_2__4_multi_spike__gru_pred.csv        